# 09 — Error Analysis

Eğitilmiş `07_full_gnn_rich` modelinin **test setindeki (2010-2016, 358 örnek)** hatalarını analiz eder.

## Analiz bölümleri
1. **Confusion matrix** — optimal F1 eşiğinde
2. **Yıla göre hata dağılımı** — hangi yıllarda model zorlanıyor?
3. **Koalisyon boyutuna göre** — büyük/küçük koalisyonlarda farklı performans?
4. **Koalisyon tipine göre** — alliance vs bloc vs de_facto
5. **Top False Positives** — model yanlışlıkla "geçerli" demiş
6. **Top False Negatives** — model gerçek koalisyonu kaçırmış
7. **Calibration** — predicted probability güvenilir mi?

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/coalition_gnn'
SNAP_DIR = os.path.join(PROJECT_DIR, 'data/snapshots')
MODELS_DIR = os.path.join(PROJECT_DIR, 'models')
CANON_DIR = os.path.join(PROJECT_DIR, 'data/canonical')
COAL_DIR = os.path.join(PROJECT_DIR, 'data/coalitions')

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import RGCNConv
from sklearn.metrics import confusion_matrix, precision_recall_curve, roc_auc_score, average_precision_score
import matplotlib.pyplot as plt
import seaborn as sns

EDGE_TYPES = ['allies', 'trades', 'votes_with', 'conflicts_with']
VAL_END = 2009

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
# Snapshot'ları yükle
def load_snapshot(year):
    path = os.path.join(SNAP_DIR, f'graph_{year}.pt')
    if not os.path.exists(path): return None
    data = torch.load(path, weights_only=False)
    x = data['country'].x; cow_codes = data['country'].cow_code.numpy()
    ei_list, et_list = [], []
    for rel_idx, rel_name in enumerate(EDGE_TYPES):
        key = ('country', rel_name, 'country')
        if key in data.edge_types:
            ei = data[key].edge_index
            ei_list.append(ei); et_list.append(torch.full((ei.shape[1],), rel_idx, dtype=torch.long))
    if not ei_list:
        edge_index = torch.zeros((2,0), dtype=torch.long); edge_type = torch.zeros(0, dtype=torch.long)
    else:
        edge_index = torch.cat(ei_list, dim=1); edge_type = torch.cat(et_list)
    return {'x': x.to(DEVICE), 'edge_index': edge_index.to(DEVICE), 'edge_type': edge_type.to(DEVICE),
            'cow_codes': cow_codes, 'code_to_idx': {int(c): i for i, c in enumerate(cow_codes)}, 'year': year}

snapshots = {}
for year in range(1946, 2017):
    s = load_snapshot(year)
    if s: snapshots[year] = s

NUM_FEATURES = next(iter(snapshots.values()))['x'].shape[1]
print(f'Snapshot: {len(snapshots)} yıl')

In [ ]:
# Test örneklerini yükle (07'deki aynı protokol)
pos_df = pd.read_parquet(os.path.join(COAL_DIR, 'positive_samples.parquet'))
neg_df = pd.read_parquet(os.path.join(COAL_DIR, 'negative_samples.parquet'))

def parse_members(s):
    if isinstance(s, list): return [int(x) for x in s]
    if isinstance(s, str): return [int(x) for x in s.split(',')]
    return []

def build_samples(df, label):
    out = []
    for _, row in df.iterrows():
        year = int(row['year'])
        if year not in snapshots: continue
        members = parse_members(row['members_str'])
        c2i = snapshots[year]['code_to_idx']
        valid = [m for m in members if m in c2i]
        if len(valid) < 2: continue
        out.append({
            'year': year, 'members': valid,
            'member_idx': [c2i[m] for m in valid],
            'label': label,
            'coalition_id': row.get('coalition_id', 'unknown') if label == 1 else f'NEG_{len(out)}',
            'coalition_name': row.get('coalition_name', '') if label == 1 else '(negative)',
        })
    return out

all_samples = build_samples(pos_df, 1) + build_samples(neg_df, 0)
test_samples = [s for s in all_samples if s['year'] > VAL_END]
print(f'Test örnek: {len(test_samples)}')
print(f'  Pozitif: {sum(1 for s in test_samples if s["label"]==1)}')
print(f'  Negatif: {sum(1 for s in test_samples if s["label"]==0)}')

In [ ]:
# 07 modelini yükle
class RGCNEncoder(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_relations, dropout=0.3):
        super().__init__()
        self.conv1 = RGCNConv(in_channels, hidden_channels, num_relations)
        self.conv2 = RGCNConv(hidden_channels, out_channels, num_relations)
        self.dropout = dropout
        self.norm1 = nn.LayerNorm(hidden_channels); self.norm2 = nn.LayerNorm(out_channels)
    def forward(self, x, ei, et):
        h = self.conv1(x, ei, et); h = self.norm1(h); h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=self.training)
        h = self.conv2(h, ei, et); h = self.norm2(h)
        return h

class RichCoalitionHead(nn.Module):
    def __init__(self, embed_dim, raw_dim, hidden_dim=128, dropout=0.3):
        super().__init__()
        pool_dim = 4 * (embed_dim + raw_dim) + 2
        self.W_pair = nn.Parameter(torch.randn(embed_dim, embed_dim) * 0.01)
        self.validity_mlp = nn.Sequential(
            nn.Linear(pool_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1))
        embed_pool_dim = 4 * embed_dim + 1
        self.duration_head = nn.Sequential(nn.Linear(embed_pool_dim, hidden_dim), nn.ReLU(),
                                            nn.Linear(hidden_dim, 1), nn.Softplus())
        self.cohesion_head = nn.Sequential(nn.Linear(embed_pool_dim, hidden_dim), nn.ReLU(),
                                            nn.Linear(hidden_dim, 1), nn.Sigmoid())
    def rich_pool(self, f):
        std = f.std(dim=0) if f.shape[0] > 1 else torch.zeros_like(f[0])
        return torch.cat([f.mean(dim=0), std, f.max(dim=0).values, f.min(dim=0).values])
    def forward(self, z, x_raw):
        k = z.shape[0]; size_feat = torch.tensor([k/20.0], device=z.device)
        z_pool = self.rich_pool(z); raw_pool = self.rich_pool(x_raw)
        if k >= 2:
            Wz = z @ self.W_pair; pm = z @ Wz.T
            pair = (pm.sum() - pm.diag().sum()) / (k*(k-1))
        else:
            pair = torch.tensor(0.0, device=z.device)
        v = self.validity_mlp(torch.cat([z_pool, raw_pool, size_feat, pair.unsqueeze(0)])).squeeze()
        mt = torch.cat([z_pool, size_feat])
        return v, self.duration_head(mt).squeeze(), self.cohesion_head(mt).squeeze()

class FullRichGNN(nn.Module):
    def __init__(self, num_features, hidden_dim=128, embed_dim=128, num_relations=4, head_hidden=128, dropout=0.3):
        super().__init__()
        self.encoder = RGCNEncoder(num_features, hidden_dim, embed_dim, num_relations, dropout)
        self.head = RichCoalitionHead(embed_dim, num_features, head_hidden, dropout)
    def forward_sample(self, snap, member_idx_list):
        z_all = self.encoder(snap['x'], snap['edge_index'], snap['edge_type'])
        idx = torch.tensor(member_idx_list, dtype=torch.long, device=DEVICE)
        return self.head(z_all[idx], snap['x'][idx])

model = FullRichGNN(NUM_FEATURES).to(DEVICE)
model.load_state_dict(torch.load(os.path.join(MODELS_DIR, 'full_rich_gnn.pt'), weights_only=False))
model.eval()
print('✓ Model yüklendi')

In [ ]:
# Tüm test örnekleri için tahminleri al
test_logits, test_labels, test_meta = [], [], []
with torch.no_grad():
    for s in test_samples:
        v, _, _ = model.forward_sample(snapshots[s['year']], s['member_idx'])
        test_logits.append(v.item())
        test_labels.append(s['label'])
        test_meta.append(s)

test_logits = np.array(test_logits)
test_labels = np.array(test_labels)
test_probs = 1 / (1 + np.exp(-test_logits))

# Optimal F1 eşiği
prec, rec, thr = precision_recall_curve(test_labels, test_probs)
f1_arr = 2*prec[:-1]*rec[:-1] / (prec[:-1]+rec[:-1]+1e-10)
best_idx = f1_arr.argmax()
best_threshold = thr[best_idx]
test_preds = (test_probs >= best_threshold).astype(int)

print(f'Optimal F1 eşiği: {best_threshold:.4f}')
print(f'Optimal F1: {f1_arr[best_idx]:.4f}')
print(f'Test AUC:   {roc_auc_score(test_labels, test_probs):.4f}')
print(f'Test PR-AUC: {average_precision_score(test_labels, test_probs):.4f}')

## 1. Confusion Matrix

In [ ]:
cm = confusion_matrix(test_labels, test_preds)
TN, FP, FN, TP = cm.ravel()

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negatif', 'Pozitif'], yticklabels=['Negatif', 'Pozitif'], ax=ax)
ax.set_xlabel('Tahmin'); ax.set_ylabel('Gerçek')
ax.set_title(f'Confusion Matrix (eşik={best_threshold:.3f})')
plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'err_confusion.png'), dpi=120)
plt.show()

print(f'TP={TP}, FP={FP}, FN={FN}, TN={TN}')
print(f'Precision = {TP/(TP+FP):.4f}')
print(f'Recall    = {TP/(TP+FN):.4f}')
print(f'FP rate   = {FP/(FP+TN):.4f}')
print(f'FN rate   = {FN/(FN+TP):.4f}')

## 2. Yıla Göre Hata Dağılımı

In [ ]:
df = pd.DataFrame({
    'year':  [s['year'] for s in test_meta],
    'label': test_labels,
    'pred':  test_preds,
    'prob':  test_probs,
    'size':  [len(s['members']) for s in test_meta],
    'correct': (test_labels == test_preds).astype(int),
})

year_stats = df.groupby('year').agg(
    n=('label', 'count'),
    acc=('correct', 'mean'),
    auc_n=('label', lambda x: x.sum())  # pozitif sayısı
).reset_index()

# Her yıl için AUC hesapla (yeterli pozitif/negatif varsa)
aucs = []
for year, sub in df.groupby('year'):
    if sub['label'].nunique() < 2:
        aucs.append(np.nan)
    else:
        aucs.append(roc_auc_score(sub['label'], sub['prob']))
year_stats['auc'] = aucs

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(year_stats['year'], year_stats['acc'], color='steelblue', alpha=0.7)
axes[0].axhline(year_stats['acc'].mean(), ls='--', color='red', label=f'Ortalama: {year_stats["acc"].mean():.3f}')
axes[0].set_title('Yıl başına Accuracy')
axes[0].set_xlabel('Yıl'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(alpha=0.3)

valid_auc = year_stats.dropna(subset=['auc'])
axes[1].bar(valid_auc['year'], valid_auc['auc'], color='seagreen', alpha=0.7)
axes[1].axhline(valid_auc['auc'].mean(), ls='--', color='red', label=f'Ortalama: {valid_auc["auc"].mean():.3f}')
axes[1].set_title('Yıl başına AUC (her iki sınıf bulunan yıllar)')
axes[1].set_xlabel('Yıl'); axes[1].set_ylabel('AUC')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'err_by_year.png'), dpi=120)
plt.show()

print(year_stats.to_string(index=False))

## 3. Koalisyon Boyutuna Göre

In [ ]:
# Boyut binleri
df['size_bin'] = pd.cut(df['size'], bins=[1, 3, 5, 10, 20, 100], labels=['2-3', '4-5', '6-10', '11-20', '21+'])

size_stats = df.groupby('size_bin', observed=True).agg(
    n=('label', 'count'),
    acc=('correct', 'mean'),
    pos_ratio=('label', 'mean'),
).reset_index()

# AUC her bin için
size_aucs = []
for sz, sub in df.groupby('size_bin', observed=True):
    if sub['label'].nunique() < 2:
        size_aucs.append(np.nan)
    else:
        size_aucs.append(roc_auc_score(sub['label'], sub['prob']))
size_stats['auc'] = size_aucs

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(size_stats['size_bin'].astype(str), size_stats['acc'], color='coral', alpha=0.7)
axes[0].set_title('Koalisyon boyutuna göre accuracy')
axes[0].set_xlabel('Üye sayısı bin'); axes[0].set_ylabel('Accuracy')
for i, n in enumerate(size_stats['n']):
    axes[0].text(i, 0.02, f'n={n}', ha='center', fontsize=9)
axes[0].grid(alpha=0.3)

axes[1].bar(size_stats['size_bin'].astype(str), size_stats['auc'], color='mediumseagreen', alpha=0.7)
axes[1].set_title('Koalisyon boyutuna göre AUC')
axes[1].set_xlabel('Üye sayısı bin'); axes[1].set_ylabel('AUC')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'err_by_size.png'), dpi=120)
plt.show()

print(size_stats.to_string(index=False))

## 4. Koalisyon Tipine Göre (canon mapping ile)

In [ ]:
# coalitions_canon.yaml'dan id → type mapping yap
import yaml
canon_path = os.path.join(PROJECT_DIR, 'coalitions_canon.yaml')
if not os.path.exists(canon_path):
    canon_path = '/content/drive/MyDrive/coalition_gnn/coalitions_canon.yaml'

try:
    with open(canon_path) as f:
        canon = yaml.safe_load(f)
    canon_types = {c['id']: c.get('type', 'unknown') for c in canon['coalitions']}
    canon_regions = {c['id']: c.get('region', 'unknown') for c in canon['coalitions']}
    print(f'Canon yüklendi: {len(canon_types)} koalisyon')
except Exception as e:
    print(f'Canon yüklenemedi: {e}')
    canon_types = {}
    canon_regions = {}

# Test örneklerinin pozitiflerini canon tipiyle eşle
df['coalition_id'] = [s['coalition_id'] for s in test_meta]
df['type'] = df['coalition_id'].map(canon_types).fillna('negative_or_unknown')
df['region'] = df['coalition_id'].map(canon_regions).fillna('negative_or_unknown')

# Sadece pozitif örnekler için tip analizi (negatifler 'negative_or_unknown')
pos_df_test = df[df['label'] == 1]

type_stats = pos_df_test.groupby('type').agg(
    n=('label', 'count'),
    recall=('correct', 'mean'),
    avg_prob=('prob', 'mean'),
).reset_index().sort_values('n', ascending=False)

print('\nPozitif test örneklerinin tip dağılımı ve recall:')
print(type_stats.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
if len(type_stats) > 0:
    ax.bar(type_stats['type'], type_stats['recall'], color='steelblue', alpha=0.7)
    for i, (n, r) in enumerate(zip(type_stats['n'], type_stats['recall'])):
        ax.text(i, r + 0.02, f'n={n}', ha='center', fontsize=9)
    ax.set_title('Koalisyon tipine göre recall (pozitif örneklerde)')
    ax.set_ylabel('Recall')
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'err_by_type.png'), dpi=120)
plt.show()

## 5. Top False Positives (model yanlışlıkla "geçerli" demiş)

Negatif örneklerden en yüksek olasılık alanlar. Model neden bunları geçerli sandı?

In [ ]:
country_master = pd.read_parquet(os.path.join(CANON_DIR, 'country_master.parquet'))
cow_to_name = {}
for _, row in country_master.iterrows():
    cow_to_name[int(row['cow_code'])] = row.get('state_name', f"COW{row['cow_code']}")

def members_str(cow_list, max_show=8):
    names = [cow_to_name.get(c, str(c)) for c in cow_list]
    if len(names) <= max_show:
        return ', '.join(names)
    return ', '.join(names[:max_show]) + f' (+{len(names)-max_show})'

neg_indices = np.where(test_labels == 0)[0]
fp_order = neg_indices[np.argsort(-test_probs[neg_indices])][:10]

print(f'{"Yıl":<6}{"k":<4}{"p(valid)":<12}{"Üyeler"}')
print('-' * 90)
for i in fp_order:
    s = test_meta[i]
    print(f'{s["year"]:<6}{len(s["members"]):<4}{test_probs[i]:<12.4f}{members_str(s["members"])}')

## 6. Top False Negatives (model gerçek koalisyonu kaçırmış)

In [ ]:
pos_indices = np.where(test_labels == 1)[0]
fn_order = pos_indices[np.argsort(test_probs[pos_indices])][:10]

print(f'{"Yıl":<6}{"k":<4}{"p(valid)":<12}{"Koalisyon":<30}{"Üyeler"}')
print('-' * 130)
for i in fn_order:
    s = test_meta[i]
    name = s.get('coalition_id', '?')
    print(f'{s["year"]:<6}{len(s["members"]):<4}{test_probs[i]:<12.4f}{name:<30}{members_str(s["members"])}')

## 7. Calibration Analysis

Predicted probability güvenilir mi? Reliability diagram.

In [ ]:
from sklearn.calibration import calibration_curve

fraction_pos, mean_pred = calibration_curve(test_labels, test_probs, n_bins=10, strategy='quantile')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Reliability diagram
axes[0].plot([0,1], [0,1], 'k--', alpha=0.5, label='Mükemmel kalibrasyon')
axes[0].plot(mean_pred, fraction_pos, 'o-', color='steelblue', markersize=8, label='Model')
axes[0].set_xlabel('Ortalama predicted probability')
axes[0].set_ylabel('Gerçek pozitif oranı')
axes[0].set_title('Reliability Diagram')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Predicted probability histogramı (label'a göre)
axes[1].hist(test_probs[test_labels==1], bins=20, alpha=0.6, color='seagreen', label='Pozitif')
axes[1].hist(test_probs[test_labels==0], bins=20, alpha=0.6, color='salmon', label='Negatif')
axes[1].axvline(best_threshold, ls='--', color='red', label=f'Eşik={best_threshold:.3f}')
axes[1].set_xlabel('Predicted probability')
axes[1].set_ylabel('Sayı')
axes[1].set_title('Olasılık dağılımları')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'err_calibration.png'), dpi=120)
plt.show()

# Brier score
from sklearn.metrics import brier_score_loss
brier = brier_score_loss(test_labels, test_probs)
print(f'Brier score: {brier:.4f} (düşük=iyi, mükemmel=0, rastgele≈0.25)')

## Özet — Makaleye Girecek Bulgular

Bu hücreyi çalıştırdıktan sonra şunları yazacağız:

1. **Confusion matrix:** Hangi tür hata baskın? (FP vs FN)
2. **Yıl analizi:** Hangi yıllarda model en çok yanılıyor? Nedeni? (örn. 2014 Kırım sonrası dağılım kayması?)
3. **Boyut analizi:** Küçük/büyük koalisyonlarda fark var mı?
4. **Tip analizi:** Alliance vs bloc vs de_facto performansı
5. **FP'ler:** Hangi negatif örnekleri model geçerli sandı? Bu örnekler aslında gerçekten gerçekçi mi?
6. **FN'ler:** Hangi gerçek koalisyonları model kaçırdı? Hangi dönemde ve neden?
7. **Calibration:** Predicted probability güvenilir mi? Brier score düzeyi.

Bu bulgular makalenin **§5 Experiments — Error Analysis** ve **§6 Discussion — Limitations** bölümlerine girecek.

## 📋 Paylaşılabilir Metin Özeti

Bu hücreyi çalıştır, çıktıyı Claude'a kopyala-yapıştır.

In [ ]:
# === 09_ERROR_ANALYSIS — PAYLAŞIM ÖZETİ ===

print('='*70)
print('ERROR ANALYSIS ÖZETİ — Claude ile paylaşmak için kopyala-yapıştır')
print('='*70)

print('\n### Genel Metrikler ###')
print(f"  Test örnek: {len(test_labels)}")
print(f"  Pozitif:    {test_labels.sum()} ({test_labels.mean():.1%})")
print(f"  Optimal F1 eşiği: {best_threshold:.4f}")
print(f"  AUC:    {roc_auc_score(test_labels, test_probs):.4f}")
print(f"  PR-AUC: {average_precision_score(test_labels, test_probs):.4f}")
print(f"  F1@opt: {f1_arr[best_idx]:.4f}")
print(f"  Brier:  {brier_score_loss(test_labels, test_probs):.4f}")

print('\n### Confusion Matrix ###')
print(f"            | Pred Neg | Pred Pos |")
print(f"  Gerçek Neg|  TN={TN:4d}  |  FP={FP:4d}  |")
print(f"  Gerçek Poz|  FN={FN:4d}  |  TP={TP:4d}  |")
print(f"  Precision = {TP/(TP+FP):.4f}, Recall = {TP/(TP+FN):.4f}")

print('\n### Yıla Göre ###')
print('| Yıl  | n  | Pozitif | Accuracy | AUC    |')
print('|------|----|---------|----------|--------|')
for _, row in year_stats.iterrows():
    auc_str = f"{row['auc']:.4f}" if not pd.isna(row['auc']) else '  -   '
    print(f"| {int(row['year']):4d} | {int(row['n']):2d} | {int(row['auc_n']):7d} | {row['acc']:.4f}   | {auc_str} |")

print('\n### Boyuta Göre ###')
print('| Boyut | n  | Pozitif% | Accuracy | AUC    |')
print('|-------|----|----------|----------|--------|')
for _, row in size_stats.iterrows():
    auc_str = f"{row['auc']:.4f}" if not pd.isna(row['auc']) else '  -   '
    print(f"| {str(row['size_bin']):5s} | {int(row['n']):2d} | {row['pos_ratio']:.3f}    | {row['acc']:.4f}   | {auc_str} |")

print('\n### Koalisyon Tipine Göre (sadece pozitif örnekler) ###')
print('| Tip               | n  | Recall | Avg p(valid) |')
print('|-------------------|----|--------|--------------|')
for _, row in type_stats.iterrows():
    print(f"| {row['type']:17s} | {int(row['n']):2d} | {row['recall']:.4f} | {row['avg_prob']:.4f}       |")

print('\n### Top 10 False Positives (model yanlışlıkla "geçerli" demiş) ###')
print('| Yıl  | k  | p(valid) | Üyeler |')
print('|------|----|----------|--------|')
for i in fp_order:
    s = test_meta[i]
    print(f"| {s['year']:4d} | {len(s['members']):2d} | {test_probs[i]:.4f}   | {members_str(s['members'])} |")

print('\n### Top 10 False Negatives (model gerçek koalisyonu kaçırmış) ###')
print('| Yıl  | k  | p(valid) | Koalisyon              | Üyeler |')
print('|------|----|----------|------------------------|--------|')
for i in fn_order:
    s = test_meta[i]
    cname = s.get('coalition_id', '?')[:22]
    print(f"| {s['year']:4d} | {len(s['members']):2d} | {test_probs[i]:.4f}   | {cname:22s} | {members_str(s['members'])} |")

print('\n### Calibration Reliability ###')
print('| Bin | Mean predicted | Fraction positive | Diff   |')
print('|-----|----------------|-------------------|--------|')
for mp, fp in zip(mean_pred, fraction_pos):
    diff = fp - mp
    sign = '+' if diff >= 0 else ''
    print(f"|  -  | {mp:.4f}         | {fp:.4f}            | {sign}{diff:.4f} |")

print('\n' + '='*70)
print('Özet tamamlandı.')
print('='*70)
